# T-shirt colours, week 3: planned stock as counts

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Goodness-of-Fit Test** · a worked example from the StatNex teaching guide

**The story.** A clothing brand ordered 4,200 T-shirts in nine colours for the season, following its stocking plan. Three weeks into the season it has sold 369 T-shirts, the first week's 120 included. The first week's check could not reject the plan, so the buyer checks again with three times the sales.

**The question.** Do sales follow the stocking plan, or did we buy the wrong colours?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## Before the analysis

In [ ]:
# Each script saves its figures to the folder figures/, as it would on your computer.
# This shows each new figure under the cell that saved it.
from pathlib import Path

import IPython.display
import matplotlib

matplotlib.use("Agg")


def show_new_figures(*_, shown=set()):
    for path in sorted(Path("figures").glob("*.png"), key=lambda p: p.stat().st_mtime_ns):
        if (path.name, path.stat().st_mtime_ns) not in shown:
            shown.add((path.name, path.stat().st_mtime_ns))
            IPython.display.display(IPython.display.Image(filename=str(path)))


events = IPython.get_ipython().events
for old in [f for f in events.callbacks["post_run_cell"] if f.__name__ == "show_new_figures"]:
    events.unregister("post_run_cell", old)
events.register("post_run_cell", show_new_figures)

## The table

This cell saves the table `04_crosstab/goodness_of_fit__week3.csv` (10 rows) as `data.csv`, the file the code reads.

In [ ]:
with open("data.csv", "w", encoding="utf-8") as file:
    file.write("""\
Colour,Units sold,Proportion,Planned stock,Planned proportion
Black,112.0,0.3035,800.0,0.1905
White,78.0,0.2114,700.0,0.1667
Navy,52.0,0.1409,500.0,0.119
Grey,46.0,0.1247,500.0,0.119
Red,24.0,0.065,400.0,0.0952
Green,18.0,0.0488,400.0,0.0952
Pastel pink,15.0,0.0407,400.0,0.0952
Mustard,9.0,0.0244,300.0,0.0714
Tie-dye,15.0,0.0407,200.0,0.0476
Total,369.0,1.0,4200.0,1.0
""")

## Helper: totals

A step that no library does, written out. In the code download it is the file `totals.py`, which the analysis imports; here the analysis uses it directly.

In [ ]:
"""Whether a row or column of a table is a margin -- a Total -- rather than a category.

A table copied from a report usually brings its Total row and column along. Read as
data, they would count every case twice, so the app leaves out each row and column
named Total or Sum, or 合計, 計, 全体 or 総計 (the Japanese words for a total). It
compares the names its own way, and so does this: full-width letters read as ordinary
ones (Unicode NFKC), every space and underscore taken out, capitals or not. "TOTAL",
"To tal" and "_Sum_" are totals; "Subtotal" and "Grand total" are not.
"""

import re
import unicodedata

TOTAL_NAMES = {"total", "sum", "合計", "計", "全体", "総計"}


def is_total(name):
    key = unicodedata.normalize("NFKC", str(name))
    key = re.sub(r"[\s_]", "", key)
    return key.lower() in TOTAL_NAMES

## Libraries

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One row per class: the observed count, and the expected count or proportion.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
names = data["Colour"].astype(str)
# A total row (Total, or its Japanese equivalent) is a margin, not a class: counting it would double every count,
# so it is left out. Spaces, underscores, case and full-width letters are ignored when
# names are compared (totals.py).
# is_total: defined in the helper cell totals above (totals.py in the code download)

keep = [not is_total(name) for name in names]
data = data[keep]
names = list(names[keep])
observed = data["Units sold"].to_numpy(float)
expected_given = data["Planned stock"].to_numpy(float)

## Goodness-of-fit test

In [ ]:
# The expected column may hold counts or proportions: either way it is rescaled to the
# observed total, because the hypothesis is about the shape, not the size.
# How to read it: each class adds (observed - expected)^2 / expected to the chi-square,
# so the contribution column shows where the fit fails. P is the chance of a total at
# least this large if the stated proportions were true. The test assumes independent
# observations and expected counts of about 5 or more in every class.
expected = expected_given / expected_given.sum() * observed.sum()
test = stats.chisquare(observed, f_exp=expected)
df = len(observed) - 1
table = pd.DataFrame({"observed": observed, "expected": expected,
                      "contribution": (observed - expected) ** 2 / expected}, index=names)
print(table)
print(f"\nchi-square = {test.statistic:.4f}, df = {df}, P = {test.pvalue:.4g}")

## Observed and expected counts

In [ ]:
Path("figures").mkdir(exist_ok=True)
figure, axis = plt.subplots(figsize=(7, 4))
positions = np.arange(1, len(names) + 1)
axis.plot(positions, observed, "o", label="observed")
axis.plot(positions, expected, "D", label="expected")
axis.set_xticks(positions)
axis.legend()
figure.tight_layout()
figure.savefig("figures/fit.png", dpi=120)

## What the result says

After week 1, the sales were too few to reject the stocking plan. Now there are 369 sales, and the goodness-of-fit test gives a different answer. Chi-square is 66.01 with 8 degrees of freedom, and the P value is far below 0.01. A P value is the chance of gaps at least this large if customers bought in the plan's proportions, so luck does not explain them.

The gaps are the ones week 1 hinted at. Black has about 30% of sales against a planned 19%. Mustard has 2.4% against 7.1%, and Pastel pink (4.1%) and Green (4.9%) are each planned at 9.5%.

So customers do not buy colours in the plan's proportions. The brand should reorder Black and expect unsold Mustard, Pastel pink and Green.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.